# Shapiro 时间延迟（T_PM）— MeTp：D 发射，C 接收（收发两端均做二阶泰勒展开）

本 notebook 计算 **MeTp 的 TPM**，对应 **M 星发射至 T 星接收** 的过程。  
按你的映射关系，这里采用：

- **M 星 = D 星 = 发射端**
- **T 星 = C 星 = 接收端**

这次按你的更正，**`r_r` 和 `r_e` 都采用二阶泰勒展开**，并且展开中使用的 \(\Delta t\) 不是 `dt_inst`，而是**原代码中的时间展开量**：

- 接收端使用 `dt_recv_corr`
- 发射端使用 `dt_emit`

即：

\[
\mathbf r_A(t_r-\varepsilon)\approx
\mathbf r_A(t_r)-\dot{\mathbf r}_A(t_r)\,\varepsilon
+\frac{1}{2}\ddot{\mathbf r}_A(t_r)\,\varepsilon^2
\]

在本问题中，对应写成：

\[
\mathbf r_r \approx
\mathbf r_C(t_r)+\mathbf v_C(t_r)\,\Delta t_{\mathrm{recv}}
+\frac{1}{2}\mathbf a_C(t_r)\,\Delta t_{\mathrm{recv}}^2
\]

其中 `dt_recv_corr` 本身通常为负值，因此上式与原代码的时间修正写法保持一致。

\[
\mathbf r_e \approx
\mathbf r_D(t_r)-\mathbf v_D(t_r)\,\Delta t_{\mathrm{emit}}
+\frac{1}{2}\mathbf a_D(t_r)\,\Delta t_{\mathrm{emit}}^2
\]

其中时间展开量沿用原代码：

\[
\Delta t_{\mathrm{inst}}=\frac{\|\mathbf r_D(t_r)-\mathbf r_C(t_r)\|}{c_0}
\]

\[
\mathbf d_0=\frac{\mathbf r_D(t_r)-\mathbf r_C(t_r)}{\|\mathbf r_D(t_r)-\mathbf r_C(t_r)\|}
\]

\[
\Delta t_{\mathrm{recv}} = -\Delta t_{\mathrm{inst}}
-\Delta t_{\mathrm{inst}}\frac{\mathbf d_0\cdot\mathbf v_C}{c_0}
\]

\[
\Delta t_{\mathrm{emit}} = \Delta t_{\mathrm{inst}}
\frac{2c_0+\mathbf d_0\cdot\mathbf v_C-\mathbf d_0\cdot\mathbf v_D}{c_0}
\]

加速度按地球点质量模型计算：

\[
a=\frac{GM_E}{R^2}
\]

程序中采用其等价向量形式：

\[
\mathbf a = -\frac{GM_E}{R^3}\mathbf r
\]

最后用修正后的发射点 `r_e` 与接收点 `r_r` 计算一程 Shapiro 延迟：

\[
T_{PM} = \frac{2GM_E}{c_0^3}
\ln\left(
\frac{\|\mathbf r_r\|+\|\mathbf r_e\|+\|\mathbf r_r-\mathbf r_e\|}
     {\|\mathbf r_r\|+\|\mathbf r_e\|-\|\mathbf r_r-\mathbf r_e\|}
\right)
\]



In [7]:
import re
import numpy as np
import pandas as pd

C0 = 299792458.0
GM_E = 3.986004418e14

GNI_C_PATH = "GNI1B_2022-06-05_C_04.txt"   # C卫星 = T星 = 接收端
GNI_D_PATH = "GNI1B_2022-06-05_D_04.txt"   # D卫星 = M星 = 发射端
OUT_CSV = "Shapiro_TPM_GNI1B_gamma_dtSR.xlsx"


In [8]:
def find_first_data_row(filepath: str) -> int:
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f):
            if re.match(r"^\s*\d+\s+", line):
                return i
    raise RuntimeError(f"No data rows found in {filepath}")


def read_gni1b(filepath: str, expected_sat: str) -> pd.DataFrame:
    cols = [
        "gps_time", "sat_id", "coord_ref",
        "x", "y", "z",
        "xerr", "yerr", "zerr",
        "vx", "vy", "vz",
        "vxerr", "vyerr", "vzerr",
        "qualflg"
    ]
    skip = find_first_data_row(filepath)
    df = pd.read_csv(filepath, sep=r"\s+", header=None, names=cols, skiprows=skip)
    df = df[(df["sat_id"] == expected_sat) & (df["coord_ref"] == "I")].copy()
    df = df[["gps_time", "x", "y", "z", "vx", "vy", "vz", "qualflg"]]
    df["gps_time"] = df["gps_time"].astype(np.int64)
    return df.sort_values("gps_time").reset_index(drop=True)


dfC = read_gni1b(GNI_C_PATH, "C")
dfD = read_gni1b(GNI_D_PATH, "D")

df = (
    dfC.merge(dfD, on="gps_time", suffixes=("_C", "_D"))
       .sort_values("gps_time")
       .reset_index(drop=True)
)

print("Merged epochs:", len(df))
df.head()


Merged epochs: 86400


,gps_time,x_C,y_C,z_C,vx_C,vy_C,vz_C,qualflg_C,x_D,y_D,z_D,vx_D,vy_D,vz_D,qualflg_D
0,707659200,3.716399e+06,3.208044e+06,4.780815e+06,-4149.229192,-3352.914250,5461.061418,10000000,3.820699e+06,3.292583e+06,4.639412e+06,-4030.053419,-3250.486988,5610.213439,10000000
1,707659201,3.712248e+06,3.204689e+06,4.786273e+06,-4153.820680,-3356.877897,5455.131498,10000000,3.816666e+06,3.289331e+06,4.645020e+06,-4034.774291,-3254.555538,5604.458237,10000000
2,707659202,3.708092e+06,3.201331e+06,4.791725e+06,-4158.407018,-3360.837382,5449.194835,10000000,3.812629e+06,3.286074e+06,4.650621e+06,-4039.490160,-3258.620051,5598.696106,10000000
3,707659203,3.703931e+06,3.197968e+06,4.797171e+06,-4162.988200,-3364.792697,5443.251438,10000000,3.808587e+06,3.282814e+06,4.656217e+06,-4044.201020,-3262.680523,5592.927053,10000000
4,707659204,3.699766e+06,3.194601e+06,4.802612e+06,-4167.564221,-3368.743840,5437.301313,10000000,3.804541e+06,3.279549e+06,4.661807e+06,-4048.906867,-3266.736949,5587.151084,10000000


In [3]:
def gravity_acceleration(r_xyz: np.ndarray, GM: float = GM_E) -> np.ndarray:
    """
    地球点质量引力模型：
        a = - GM / R^3 * r
    其模长为：
        |a| = GM / R^2
    """
    r_norm = float(np.linalg.norm(r_xyz))
    if r_norm == 0.0:
        raise ValueError("Position norm is zero; cannot compute gravity acceleration.")
    return -(GM / r_norm**3) * r_xyz


def shapiro_delay_oneway(r_e: np.ndarray, r_r: np.ndarray,
                         GM: float = GM_E, c: float = C0) -> float:
    re_norm = float(np.linalg.norm(r_e))
    rr_norm = float(np.linalg.norm(r_r))
    R_er = float(np.linalg.norm(r_r - r_e))
    num = rr_norm + re_norm + R_er
    den = rr_norm + re_norm - R_er
    if den <= 0.0:
        return np.nan
    return (2.0 * GM / (c**3)) * float(np.log(num / den))


def build_me_tp_geometry(
    r_C_tr: np.ndarray,  # C卫星 = T星 = 接收端，在 t_r 的位置
    v_C_tr: np.ndarray,  # C卫星 = T星 = 接收端，在 t_r 的速度
    r_D_tr: np.ndarray,  # D卫星 = M星 = 发射端，在 t_r 的位置
    v_D_tr: np.ndarray,  # D卫星 = M星 = 发射端，在 t_r 的速度
    c: float = C0,
):
    """
    MeTp 的 TPM：M 发射 -> T 接收
    映射为：D 发射 -> C 接收

    这里同时对接收点 r_r 和发射点 r_e 做二阶泰勒展开，
    且展开时间量沿用原代码中的 dt_recv_corr 和 dt_emit，
    而不是直接使用 dt_inst。

    d0 的定义也保持原代码风格：
        d0 = (r_M - r_T) / |r_M - r_T|
           = (r_D - r_C) / |r_D - r_C|

    原代码时间展开量：
        dt_inst      = |r_D - r_C| / c
        dt_recv_corr = -dt_inst - dt_inst * (d0·v_C) / c
        dt_emit      =  dt_inst * (2c + d0·v_C - d0·v_D) / c

    二阶泰勒展开：
        r_r ≈ r_C + v_C * dt_recv_corr + 0.5 * a_C * dt_recv_corr^2
        r_e ≈ r_D - v_D * dt_emit      + 0.5 * a_D * dt_emit^2
    """
    dr_DC = r_D_tr - r_C_tr
    rho0 = float(np.linalg.norm(dr_DC))
    if rho0 == 0.0:
        raise ValueError("r_D_tr and r_C_tr are identical, cannot define propagation direction.")

    d0 = dr_DC / rho0
    dt_inst = rho0 / c

    d0_dot_vC = float(np.dot(d0, v_C_tr))
    d0_dot_vD = float(np.dot(d0, v_D_tr))

    dt_recv_corr = -dt_inst - dt_inst * (d0_dot_vC / c)
    dt_emit = dt_inst * (2.0 * c + d0_dot_vC - d0_dot_vD) / c

    a_C_tr = gravity_acceleration(r_C_tr, GM=GM_E)
    a_D_tr = gravity_acceleration(r_D_tr, GM=GM_E)

    r_r = r_C_tr + v_C_tr * dt_recv_corr + 0.5 * a_C_tr * (dt_recv_corr ** 2)
    r_e = r_D_tr - v_D_tr * dt_emit + 0.5 * a_D_tr * (dt_emit ** 2)

    return {
        "r_e": r_e,
        "r_r": r_r,
        "dt_inst": dt_inst,
        "dt_emit": dt_emit,
        "dt_recv_corr": dt_recv_corr,
        "d0": d0,
        "d0_dot_vC": d0_dot_vC,
        "d0_dot_vD": d0_dot_vD,
        "a_C_tr": a_C_tr,
        "a_D_tr": a_D_tr,
    }


In [5]:
tpm = np.empty(len(df), dtype=float)
rhopm = np.empty(len(df), dtype=float)
dt_inst_arr = np.empty(len(df), dtype=float)
dt_emit_arr = np.empty(len(df), dtype=float)
dt_recv_corr_arr = np.empty(len(df), dtype=float)

d0_x = np.empty(len(df), dtype=float)
d0_y = np.empty(len(df), dtype=float)
d0_z = np.empty(len(df), dtype=float)
d0_dot_vC_arr = np.empty(len(df), dtype=float)
d0_dot_vD_arr = np.empty(len(df), dtype=float)

aC_x = np.empty(len(df), dtype=float)
aC_y = np.empty(len(df), dtype=float)
aC_z = np.empty(len(df), dtype=float)
aC_mag = np.empty(len(df), dtype=float)

aD_x = np.empty(len(df), dtype=float)
aD_y = np.empty(len(df), dtype=float)
aD_z = np.empty(len(df), dtype=float)
aD_mag = np.empty(len(df), dtype=float)

for i, row in enumerate(df.itertuples(index=False)):
    # C = T = 接收端
    r_C_tr = np.array([row.x_C, row.y_C, row.z_C], dtype=float)
    v_C_tr = np.array([row.vx_C, row.vy_C, row.vz_C], dtype=float)

    # D = M = 发射端
    r_D_tr = np.array([row.x_D, row.y_D, row.z_D], dtype=float)
    v_D_tr = np.array([row.vx_D, row.vy_D, row.vz_D], dtype=float)

    geo = build_me_tp_geometry(
        r_C_tr=r_C_tr,
        v_C_tr=v_C_tr,
        r_D_tr=r_D_tr,
        v_D_tr=v_D_tr,
        c=C0,
    )

    tpm[i] = shapiro_delay_oneway(geo["r_e"], geo["r_r"], GM=GM_E, c=C0)
    rhopm[i] = C0 * tpm[i]

    dt_inst_arr[i] = geo["dt_inst"]
    dt_emit_arr[i] = geo["dt_emit"]
    dt_recv_corr_arr[i] = geo["dt_recv_corr"]

    d0_x[i], d0_y[i], d0_z[i] = geo["d0"]
    d0_dot_vC_arr[i] = geo["d0_dot_vC"]
    d0_dot_vD_arr[i] = geo["d0_dot_vD"]

    aC_x[i], aC_y[i], aC_z[i] = geo["a_C_tr"]
    aC_mag[i] = float(np.linalg.norm(geo["a_C_tr"]))

    aD_x[i], aD_y[i], aD_z[i] = geo["a_D_tr"]
    aD_mag[i] = float(np.linalg.norm(geo["a_D_tr"]))

df_out = pd.DataFrame({
    "gps_time": df["gps_time"].values,
    "T_PM_s": tpm,
    "rho_PM_m": rhopm,
    "dt_inst_s": dt_inst_arr,
    "dt_emit_s": dt_emit_arr,
    "dt_recv_corr_s": dt_recv_corr_arr,
    "d0_x": d0_x,
    "d0_y": d0_y,
    "d0_z": d0_z,
    "d0_dot_vC_mps": d0_dot_vC_arr,
    "d0_dot_vD_mps": d0_dot_vD_arr,
    "aC_x_mps2": aC_x,
    "aC_y_mps2": aC_y,
    "aC_z_mps2": aC_z,
    "aC_mag_mps2": aC_mag,
    "aD_x_mps2": aD_x,
    "aD_y_mps2": aD_y,
    "aD_z_mps2": aD_z,
    "aD_mag_mps2": aD_mag,
})

df_out.to_excel(OUT_CSV, index=False)
print("Saved:", OUT_CSV)
df_out.head()


Saved: Shapiro_TPM_GNI1B_MeTp_D_emit_C_recv_taylor2_both_ends.xlsx


,gps_time,T_PM_s,rho_PM_m,dt_inst_s,dt_emit_s,dt_recv_corr_s,d0_x,d0_y,d0_z,d0_dot_vC_mps,d0_dot_vD_mps,aC_x_mps2,aC_y_mps2,aC_z_mps2,aC_mag_mps2,aD_x_mps2,aD_y_mps2,aD_z_mps2,aD_mag_mps2
0,707659200,8.419424e-13,0.000252,0.00065,0.001301,-0.00065,0.534905,0.433562,-0.725190,-7633.447095,-7633.454466,-4.603380,-3.973698,-5.921836,8.488199,-4.732024,-4.077941,-5.746020,8.487544
1,707659201,8.419436e-13,0.000252,0.00065,0.001301,-0.00065,0.535515,0.434087,-0.724425,-7633.446221,-7633.454250,-4.598258,-3.969559,-5.928623,8.488224,-4.727052,-4.073931,-5.752991,8.487570
2,707659202,8.419448e-13,0.000252,0.00065,0.001301,-0.00065,0.536126,0.434612,-0.723659,-7633.445323,-7633.454009,-4.593130,-3.965416,-5.935402,8.488248,-4.722073,-4.069917,-5.759955,8.487596
3,707659203,8.419461e-13,0.000252,0.00065,0.001301,-0.00065,0.536735,0.435136,-0.722892,-7633.444399,-7633.453743,-4.587996,-3.961268,-5.942174,8.488273,-4.717089,-4.065897,-5.766912,8.487622
4,707659204,8.419473e-13,0.000252,0.00065,0.001301,-0.00065,0.537344,0.435659,-0.722124,-7633.443450,-7633.453451,-4.582856,-3.957114,-5.948938,8.488297,-4.712099,-4.061872,-5.773862,8.487648


In [6]:
print("T_PM (s): min / mean / max =",
      np.nanmin(df_out["T_PM_s"]),
      np.nanmean(df_out["T_PM_s"]),
      np.nanmax(df_out["T_PM_s"]))

print("rho_PM (m): min / mean / max =",
      np.nanmin(df_out["rho_PM_m"]),
      np.nanmean(df_out["rho_PM_m"]),
      np.nanmax(df_out["rho_PM_m"]))

print("rho_PM (um): min / mean / max =",
      1e6 * np.nanmin(df_out["rho_PM_m"]),
      1e6 * np.nanmean(df_out["rho_PM_m"]),
      1e6 * np.nanmax(df_out["rho_PM_m"]))


T_PM (s): min / mean / max = 8.327204259932064e-13 8.375930081520036e-13 8.421376980533153e-13
rho_PM (m): min / mean / max = 0.00024964330333531045 0.00025110406671750314 0.0002524665304738652
rho_PM (um): min / mean / max = 249.64330333531046 251.10406671750314 252.46653047386522


In [ ]:
# 如需查看前几个历元结果，可运行：
df_out.head(10)
